# Django, Migrations

## Introduction to Django Migrations
---

In this lesson, you'll learn how **migrations** work in Django.

Migrations are Django's way of propagating changes you make to your models into your database schema.

1. [What are Migrations](#What-are-Migrations),
    - [Why Migrations Matter](#Why-Migrations-Matter),
    - [The Migration Workflow](#The-Migration-Workflow),
2. [Creating Migrations with makemigrations](#Creating-Migrations-with-makemigrations),
    - [Running makemigrations](#Running-makemigrations),
    - [What Gets Created](#What-Gets-Created),
3. [Applying Migrations with migrate](#Applying-Migrations-with-migrate),
    - [Running migrate](#Running-migrate),
    - [Migration Status with showmigrations](#Migration-Status-with-showmigrations),
4. [Working with Migration Files](#Working-with-Migration-Files),
    - [Understanding Migration File Structure](#Understanding-Migration-File-Structure),
    - [Version Control and Migrations](#Version-Control-and-Migrations),
5. [Troubleshooting Migrations](#Troubleshooting-Migrations),
    - [Rolling Back Migrations](#Rolling-Back-Migrations),
    - [Resolving Migration Conflicts](#Resolving-Migration-Conflicts),
    - [Fake Migrations](#Fake-Migrations),
6. [🧠 EXERCISE 🧠, Create and Apply Migrations](#🧠-EXERCISE-🧠,-Create-and-Apply-Migrations).

<br>

## What are Migrations

---

### Why Migrations Matter

---

Imagine you've been running your bookstore app for months. The database has thousands of books and users.

Now you need to add a new field - `page_count` - to your Book model.

<br>

**Without migrations**, you would have to:
1. Write SQL manually: `ALTER TABLE books ADD COLUMN page_count INTEGER;`
2. Keep track of all changes across development, staging, and production
3. Remember which changes were applied where
4. Hope nothing breaks!

<br>

**With Django migrations**, you just:
1. Change your model in Python
2. Run `makemigrations` - Django detects the change
3. Run `migrate` - Django applies the change
4. The migration file is version controlled - same change everywhere!

<br>

**Migrations provide:**

| Benefit | Description |
|---------|-------------|
| **Version control** | Track database changes like code changes |
| **Reproducibility** | Same migrations = same schema everywhere |
| **Reversibility** | Roll back changes when needed |
| **Team collaboration** | Share database changes through Git |
| **No SQL needed** | Django generates SQL automatically |

<br>

### The Migration Workflow

---

The migration process has **three steps**:

<br>

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│   1. CHANGE     │     │   2. MAKE        │     │   3. APPLY      │
│   models.py     │ ──► │   makemigrations │ ──► │   migrate       │
│                 │     │                  │     │                 │
│   (Python code) │     │   (Creates file) │     │   (Updates DB)  │
└─────────────────┘     └──────────────────┘     └─────────────────┘
```

<br>

1. **Change**: You modify `models.py` (add field, change field, delete model, etc.)
2. **Make**: `makemigrations` detects changes and creates a migration file
3. **Apply**: `migrate` executes the migration and updates the database

<br>

## Creating Migrations with makemigrations

---

### Running makemigrations

---

The `makemigrations` command scans your models, compares them to the current migration state, and creates new migration files for any changes.

<br>

**Basic usage:**

```bash
# Create migrations for all apps
python manage.py makemigrations

# Create migrations for specific app
python manage.py makemigrations catalog

# Create migration with a custom name
python manage.py makemigrations catalog --name add_page_count_to_book
```

<br>

**Example scenario:**

Let's say you have this model:

```python
# catalog/models.py (BEFORE)

class Book(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
```

And you add a new field:

```python
# catalog/models.py (AFTER)

class Book(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    page_count = models.IntegerField(default=0)  # NEW FIELD
```

<br>

**Running makemigrations:**

```bash
$ python manage.py makemigrations catalog

Migrations for 'catalog':
  catalog/migrations/0002_book_page_count.py
    - Add field page_count to book
```

Django detected the new field and created a migration file!

<br>

### What Gets Created

---

Migration files are created in the `migrations` directory of your app:

```
catalog/
├── __init__.py
├── models.py
├── views.py
└── migrations/
    ├── __init__.py
    ├── 0001_initial.py           # First migration (creates model)
    └── 0002_book_page_count.py   # Second migration (adds field)
```

<br>

**Migration naming convention:**

| Pattern | Meaning |
|---------|---------|
| `0001_initial.py` | First migration, creates the model |
| `0002_book_page_count.py` | Second migration, auto-named based on change |
| `0003_custom_name.py` | Migration with custom name (--name flag) |

<br>

The number prefix ensures migrations are applied in order.

<br>

**Useful makemigrations options:**

```bash
# Preview what would happen (don't create files)
python manage.py makemigrations --dry-run

# See the SQL that would be generated
python manage.py sqlmigrate catalog 0002

# Check for model changes without creating migrations
python manage.py makemigrations --check
# Returns exit code 1 if changes detected (useful in CI)
```

<br>

## Applying Migrations with migrate

---

### Running migrate

---

The `migrate` command applies pending migrations to your database.

<br>

**Basic usage:**

```bash
# Apply all pending migrations
python manage.py migrate

# Apply migrations for specific app
python manage.py migrate catalog

# Apply up to a specific migration
python manage.py migrate catalog 0002
```

<br>

**Example output:**

```bash
$ python manage.py migrate

Operations to perform:
  Apply all migrations: admin, auth, catalog, contenttypes, sessions
Running migrations:
  Applying catalog.0002_book_page_count... OK
```

<br>

Django tracks which migrations have been applied in a special table called `django_migrations`.

<br>

### Migration Status with showmigrations

---

The `showmigrations` command shows the status of all migrations:

```bash
$ python manage.py showmigrations

admin
 [X] 0001_initial
 [X] 0002_logentry_remove_auto_add
 [X] 0003_logentry_add_action_flag_choices
auth
 [X] 0001_initial
 [X] 0002_alter_permission_name_max_length
 ...
catalog
 [X] 0001_initial
 [X] 0002_book_page_count
 [ ] 0003_add_publisher              # <-- Not yet applied
```

<br>

| Symbol | Meaning |
|--------|--------|
| `[X]` | Migration has been applied |
| `[ ]` | Migration is pending |

<br>

**Useful options:**

```bash
# Show only for specific app
python manage.py showmigrations catalog

# Show in plan format (order of execution)
python manage.py showmigrations --plan
```

<br>

## Working with Migration Files

---

### Understanding Migration File Structure

---

Let's look at what's inside a migration file:

In [ ]:
# Example: catalog/migrations/0002_book_page_count.py

from django.db import migrations, models


class Migration(migrations.Migration):
    
    # Dependencies - which migrations must be applied first
    dependencies = [
        ('catalog', '0001_initial'),
    ]
    
    # Operations - what this migration does
    operations = [
        migrations.AddField(
            model_name='book',
            name='page_count',
            field=models.IntegerField(default=0),
        ),
    ]

<br>

**Key components:**

| Component | Description |
|-----------|-------------|
| `dependencies` | List of migrations that must run first |
| `operations` | List of changes to apply |

<br>

**Common operations:**

| Operation | What it does |
|-----------|-------------|
| `CreateModel` | Create a new table |
| `DeleteModel` | Delete a table |
| `AddField` | Add a column |
| `RemoveField` | Remove a column |
| `AlterField` | Change a column |
| `RenameField` | Rename a column |
| `AddIndex` | Create an index |
| `RemoveIndex` | Remove an index |

<br>

### Version Control and Migrations

---

**Always commit migration files to Git!**

Migrations are part of your codebase - they ensure every environment (development, staging, production) has the same database schema.

<br>

```bash
# After creating migrations
git add catalog/migrations/
git commit -m "Add page_count field to Book model"
```

<br>

**What to commit:**

| Include | Exclude |
|---------|--------|
| Migration files (`*.py`) | `__pycache__` |
| `__init__.py` in migrations | Database files (`.sqlite3`) |

<br>

**Typical workflow with Git:**

```bash
# 1. Pull latest code (includes migrations from teammates)
git pull origin main

# 2. Apply any new migrations
python manage.py migrate

# 3. Make your model changes
# ... edit models.py ...

# 4. Create migration for your changes
python manage.py makemigrations

# 5. Test locally
python manage.py migrate
python manage.py runserver

# 6. Commit everything
git add .
git commit -m "Add new feature with model changes"
git push
```

<br>

## Troubleshooting Migrations

---

### Rolling Back Migrations

---

Made a mistake? You can **roll back** to a previous migration:

```bash
# Roll back to a specific migration
python manage.py migrate catalog 0001

# This will REVERSE all migrations after 0001
```

<br>

**Example:**

```bash
$ python manage.py showmigrations catalog
catalog
 [X] 0001_initial
 [X] 0002_book_page_count      # We want to undo this
 [X] 0003_book_publisher

$ python manage.py migrate catalog 0001
Operations to perform:
  Target specific migration: 0001_initial, from catalog
Running migrations:
  Rendering model states... DONE
  Unapplying catalog.0003_book_publisher... OK
  Unapplying catalog.0002_book_page_count... OK

$ python manage.py showmigrations catalog
catalog
 [X] 0001_initial
 [ ] 0002_book_page_count      # Now unapplied
 [ ] 0003_book_publisher
```

<br>

**To completely unapply all migrations for an app:**

```bash
python manage.py migrate catalog zero
```

⚠️ **Warning:** This removes all tables for that app!

<br>

### Resolving Migration Conflicts

---

**Scenario:** Two developers create migrations at the same time:

```
Developer A creates: 0002_add_page_count.py
Developer B creates: 0002_add_publisher.py

Both depend on 0001_initial.py!
```

<br>

When you pull both changes, Django will complain:

```
CommandError: Conflicting migrations detected; multiple leaf nodes in the 
migration graph: (0002_add_page_count, 0002_add_publisher in catalog).
```

<br>

**Solution 1: Merge migrations (recommended)**

```bash
python manage.py makemigrations --merge
```

This creates a merge migration:

```python
# 0003_merge_20260302_1430.py

class Migration(migrations.Migration):
    dependencies = [
        ('catalog', '0002_add_page_count'),
        ('catalog', '0002_add_publisher'),
    ]
    operations = []  # No operations, just connects the two branches
```

<br>

**Solution 2: Recreate one migration**

If the conflict is on your local branch (not yet pushed):

```bash
# 1. Delete your migration file
rm catalog/migrations/0002_add_page_count.py

# 2. Pull the other migration
git pull

# 3. Apply the pulled migration
python manage.py migrate

# 4. Recreate your migration (will be 0003 now)
python manage.py makemigrations
```

<br>

### Fake Migrations

---

Sometimes the database already has the changes, but Django doesn't know it. Use `--fake` to mark a migration as applied without running it:

```bash
# Mark migration as applied without executing it
python manage.py migrate catalog 0002 --fake
```

<br>

**Common use cases:**

| Scenario | Solution |
|----------|----------|
| Database was modified manually | `--fake` the migration |
| Starting fresh with existing schema | `--fake-initial` |
| Fixing migration state after manual fix | `--fake` specific migration |

<br>

**Example: New project with existing database**

```bash
# Tables already exist in database
# Mark initial migrations as applied without creating tables
python manage.py migrate --fake-initial
```

<br>

## Common Migration Scenarios

---

### Adding a Required Field to Existing Data

---

When you add a required field to a model that already has data, Django will ask what to do:

```bash
$ python manage.py makemigrations

You are trying to add a non-nullable field 'isbn' to book without a default.
We can't do that (the database needs something to populate existing rows).
Please select a fix:
 1) Provide a one-off default now (will be set on all existing rows)
 2) Quit, and let me add a default in models.py
Select an option:
```

<br>

**Options:**

1. **Option 1:** Enter a value now (e.g., `'unknown'` for a CharField)
2. **Option 2:** Add `default=...` or `null=True` in models.py first

<br>

**Best practice:** Add the field as nullable first, then populate data, then make it required:

```python
# Step 1: Add as nullable
isbn = models.CharField(max_length=13, null=True, blank=True)
# makemigrations + migrate

# Step 2: Populate existing rows (via shell or data migration)

# Step 3: Make it required
isbn = models.CharField(max_length=13, unique=True)
# makemigrations + migrate
```

<br>

### Renaming a Field

---

Django is smart about detecting field renames:

```python
# Before
class Book(models.Model):
    name = models.CharField(max_length=200)

# After
class Book(models.Model):
    title = models.CharField(max_length=200)  # Renamed
```

```bash
$ python manage.py makemigrations

Did you rename book.name to book.title (a CharField)? [y/N] y

Migrations for 'catalog':
  catalog/migrations/0004_rename_name_book_title.py
    - Rename field name on book to title
```

Django will use `RenameField` operation, preserving your data!

<br>

## Summary

---

| Command | Description |
|---------|-------------|
| `makemigrations` | Create migration files from model changes |
| `migrate` | Apply migrations to database |
| `showmigrations` | Show migration status |
| `sqlmigrate` | Show SQL for a migration |
| `migrate <app> <migration>` | Migrate to specific version |
| `migrate --fake` | Mark as applied without running |
| `makemigrations --merge` | Merge conflicting migrations |

<br>

**Key takeaways:**

- Always run `makemigrations` after changing models
- Always commit migration files to version control
- Run `migrate` on every environment (dev, staging, production)
- Use `showmigrations` to check status
- Handle conflicts with `--merge` or by recreating

<br>

### 🧠 EXERCISE 🧠, Create and Apply Migrations

---

Practice the migration workflow:

1. Create a new model `Publisher` in your catalog app with:
   - `name` (CharField, max 200)
   - `website` (URLField, optional)
   - `founded_year` (IntegerField)

2. Create a migration for the new model

3. View the SQL that would be executed (don't apply yet)

4. Apply the migration

5. Add a `publisher` ForeignKey field to the `Book` model (optional, nullable)

6. Create and apply the migration for this change

7. Check migration status with `showmigrations`

<details>
    <summary>▶️ Solution</summary>
    
**1. Edit `catalog/models.py`:**

```python
class Publisher(models.Model):
    name = models.CharField(max_length=200)
    website = models.URLField(blank=True)
    founded_year = models.IntegerField()
    
    def __str__(self):
        return self.name
```

**2. Create migration:**

```bash
python manage.py makemigrations catalog
# Output: catalog/migrations/0002_publisher.py
```

**3. View SQL:**

```bash
python manage.py sqlmigrate catalog 0002
```

**4. Apply migration:**

```bash
python manage.py migrate catalog
```

**5. Add to Book model:**

```python
class Book(models.Model):
    # ... existing fields ...
    publisher = models.ForeignKey(
        Publisher,
        on_delete=models.SET_NULL,
        null=True,
        blank=True,
        related_name='books'
    )
```

**6. Create and apply:**

```bash
python manage.py makemigrations catalog
python manage.py migrate catalog
```

**7. Check status:**

```bash
python manage.py showmigrations catalog
# catalog
#  [X] 0001_initial
#  [X] 0002_publisher
#  [X] 0003_book_publisher
```
</details>

<br>

### 🧠 EXERCISE 🧠, Handle Migration Rollback

---

Practice rolling back migrations:

1. Check current migration status with `showmigrations`

2. Roll back the last migration you applied

3. Verify the rollback with `showmigrations`

4. Re-apply the migration

5. (Optional) Try rolling back to `zero` and then migrating back up

<details>
    <summary>▶️ Solution</summary>
    
```bash
# 1. Check current status
python manage.py showmigrations catalog
# catalog
#  [X] 0001_initial
#  [X] 0002_publisher
#  [X] 0003_book_publisher

# 2. Roll back to 0002 (unapplies 0003)
python manage.py migrate catalog 0002

# 3. Verify
python manage.py showmigrations catalog
# catalog
#  [X] 0001_initial
#  [X] 0002_publisher
#  [ ] 0003_book_publisher    <-- Now unapplied

# 4. Re-apply
python manage.py migrate catalog

# 5. (Optional) Roll back everything and re-apply
python manage.py migrate catalog zero
python manage.py showmigrations catalog  # All should be [ ]
python manage.py migrate catalog
python manage.py showmigrations catalog  # All should be [X]
```
</details>

---